In [736]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
from math import sqrt
import ROOT
import ctypes
try:
#     plt.style.use('belle2')
    plt.style.use('belle2_serif')
#     plt.style.use('belle2_modern')
except OSError:
    print("Please install belle2 matplotlib style") 
px = 1/plt.rcParams['figure.dpi']

from main.data_tools.extract_ntuples import get_pd, get_np
from main.draw_tools.decorations import b2helix, watermark
from main.draw_tools.stacking_with_error_bars import MC_stack_plot, MC_stack_plot_density

from main.data_tools.error_bars import make_data_weight
from main.data_tools.query_dataframes import cut_dfs_7types

from matplotlib.ticker import ScalarFormatter


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [737]:
from math import sqrt

# Error-weighted combination function
def combine_error_weighted(x, y, x_err, y_err):
    central_value = (x / x_err**2 + y / y_err**2) / (1 / x_err**2 + 1 / y_err**2)
    error = 1 / sqrt(1 / x_err**2 + 1 / y_err**2)
    return central_value, error

In [738]:
def combine_x_plus_y_divided_by_2(x, y, x_err, y_err):
    central_value = (x+y)/2
    error = sqrt(x_err**2 +  y_err**2)/2
    return central_value, error

In [739]:
def correct_Acp_stats( Araw, Araw_err, Aref, Aref_err, Aref_pdg, Aref_K_mix):
    final_Acp = Araw - Aref + Aref_pdg + Aref_K_mix
    final_Acp_err = sqrt(Araw_err**2 + Aref_err**2)

    return final_Acp, final_Acp_err

In [740]:
def correct_Acp_stats_no_Kmix( Araw, Araw_err, Aref, Aref_err, Aref_pdg):
    final_Acp = Araw - Aref + Aref_pdg 
    final_Acp_err = sqrt(Araw_err**2 + Aref_err**2)

    return final_Acp, final_Acp_err

In [741]:
def delta_Acp_sys_unc(A_original, A_original_error, A, A_error):
    delta_Acp = A - A_original
    if A_original_error > A_error:
        delta_Acp_error = sqrt(A_original_error**2 - A_error**2)
    elif A_original_error < A_error:
        delta_Acp_error = sqrt(A_error**2 - A_original_error**2)
    else: 
        print("Error: unable to proceed.")
        sys.exit()

    # print(f"delta_Acp: {delta_Acp}, delta_Acp_error: {delta_Acp_error}")
    print(f"Original Acp: {A_original * 100:.5f}%, Original Acp error: {A_original_error * 100:.5f}%")
    print(f"Acp: {A * 100:.5f}%, Acp error: {A_error * 100:.5f}%")
    print(f"delta_Acp: {delta_Acp * 100:.5f}%, delta_Acp_error: {delta_Acp_error * 100:.5f}%")
    return delta_Acp, delta_Acp_error

In [742]:
def combine_error_weighted_3(x, y, z, x_err, y_err, z_err):
    # 각 오차의 제곱의 역수 (가중치) 합
    sum_inv_var = (1 / x_err**2) + (1 / y_err**2) + (1 / z_err**2)
    
    # Error-weighted 중앙값 계산
    central_value = (x / x_err**2 + y / y_err**2 + z / z_err**2) / sum_inv_var
    
    # 합성된 오차 계산
    error = 1 / sqrt(sum_inv_var)
    
    # 결과 출력
    print(f"val 1 = {x}, val 2 = {y}, val 3 = {z}")
    print(f"central value = {central_value*100:.4f}% \pm {error*100:.4f}%")
    
    return central_value, error


# MC15 full

## Acp(D+ -> eta pi+)

### eta -> gg

In [743]:
#bin a, f

In [744]:
#fit_v12
Araw_gg_cms_plus = 0.032568692936323806
Araw_gg_cms_plus_error =  0.01785226639611848
Araw_gg_cms_minus = -0.010052866289589013
Araw_gg_cms_minus_error = 0.01108094106790564
Araw_gg , Araw_gg_stats_error= combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg: {Araw_gg}, Araw_gg_stats_error: {Araw_gg_stats_error}")
print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

Araw_gg: 0.011257913323367397, Araw_gg_stats_error: 0.010505839690719382
Araw_gg: 1.12579%, Araw_gg_stats_error: 1.05058%


In [745]:
#fit_v3
Aref_gg_cms_plus = 0.012767243523722938
Aref_gg_cms_plus_error = 0.0035817861575496578
Aref_gg_cms_minus = -0.009112314113025932
Aref_gg_cms_minus_error = 0.0026362813045371866
Aref_gg, Aref_gg_stats_error = combine_x_plus_y_divided_by_2(Aref_gg_cms_plus,Aref_gg_cms_minus, Aref_gg_cms_plus_error,Aref_gg_cms_minus_error )

print(f"Aref_gg: {Aref_gg}, Aref_gg_stats_error: {Aref_gg_stats_error}")
print(f"Aref_gg: {Aref_gg * 100:.5f}%, Aref_gg_stats_error: {Aref_gg_stats_error * 100:.5f}%")

Aref_gg: 0.0018274647053485027, Aref_gg_stats_error: 0.002223689006755814
Aref_gg: 0.18275%, Aref_gg_stats_error: 0.22237%


In [746]:
Aref_gg_pdg = 0
Acp_etapip_gg_value_1, Acp_etapip_gg_error_1 = correct_Acp_stats_no_Kmix(Araw_gg, Araw_gg_stats_error, Aref_gg, Aref_gg_stats_error, Aref_gg_pdg)

print(f"Acp_etapip_gg_value_1: {Acp_etapip_gg_value_1 * 100:.5f}%, Acp_etapip_gg_error_1: {Acp_etapip_gg_error_1 * 100:.5f}%")

Acp_etapip_gg_value_1: 0.94304%, Acp_etapip_gg_error_1: 1.07386%


In [747]:
# bin b, e

In [748]:
#fit_v12
Araw_gg_cms_plus = -0.00762180891179276
Araw_gg_cms_plus_error =  0.008928769051545314
Araw_gg_cms_minus = 0.016609768794683122
Araw_gg_cms_minus_error = 0.00770063402795162
Araw_gg , Araw_gg_stats_error= combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg: {Araw_gg}, Araw_gg_stats_error: {Araw_gg_stats_error}")
print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

Araw_gg: 0.004493979941445181, Araw_gg_stats_error: 0.005895393990402164
Araw_gg: 0.44940%, Araw_gg_stats_error: 0.58954%


In [749]:
#fit_v3
Aref_gg_cms_plus = 0.010040423453906655
Aref_gg_cms_plus_error = 0.0022692912068116014
Aref_gg_cms_minus = -0.007031434843011053
Aref_gg_cms_minus_error = 0.002094765417047547
Aref_gg, Aref_gg_stats_error = combine_x_plus_y_divided_by_2(Aref_gg_cms_plus,Aref_gg_cms_minus, Aref_gg_cms_plus_error,Aref_gg_cms_minus_error )

print(f"Aref_gg: {Aref_gg}, Aref_gg_stats_error: {Aref_gg_stats_error}")
print(f"Aref_gg: {Aref_gg * 100:.5f}%, Aref_gg_stats_error: {Aref_gg_stats_error * 100:.5f}%")

Aref_gg: 0.001504494305447801, Aref_gg_stats_error: 0.0015441603490061221
Aref_gg: 0.15045%, Aref_gg_stats_error: 0.15442%


In [750]:
Aref_gg_pdg = 0
Acp_etapip_gg_value_2, Acp_etapip_gg_error_2 = correct_Acp_stats_no_Kmix(Araw_gg, Araw_gg_stats_error, Aref_gg, Aref_gg_stats_error, Aref_gg_pdg)

print(f"Acp_etapip_gg_value_2: {Acp_etapip_gg_value_2 * 100:.5f}%, Acp_etapip_gg_error_2: {Acp_etapip_gg_error_2 * 100:.5f}%")

Acp_etapip_gg_value_2: 0.29895%, Acp_etapip_gg_error_2: 0.60943%


In [751]:
# bin c, d

In [752]:
#fit_v12
Araw_gg_cms_plus = 0.014421193029585888
Araw_gg_cms_plus_error =  0.0068051743282280044
Araw_gg_cms_minus = 0.001643590697615327
Araw_gg_cms_minus_error = 0.0067096690895199745
Araw_gg , Araw_gg_stats_error= combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg: {Araw_gg}, Araw_gg_stats_error: {Araw_gg_stats_error}")
print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

Araw_gg: 0.008032391863600608, Araw_gg_stats_error: 0.004778338019867192
Araw_gg: 0.80324%, Araw_gg_stats_error: 0.47783%


In [753]:
#fit_v3
Aref_gg_cms_plus = 0.007484012201581125
Aref_gg_cms_plus_error = 0.0019024365813576668
Aref_gg_cms_minus = -0.0010830907936497658
Aref_gg_cms_minus_error = 0.0018983786466550427
Aref_gg, Aref_gg_stats_error = combine_x_plus_y_divided_by_2(Aref_gg_cms_plus,Aref_gg_cms_minus, Aref_gg_cms_plus_error,Aref_gg_cms_minus_error )

print(f"Aref_gg: {Aref_gg}, Aref_gg_stats_error: {Aref_gg_stats_error}")
print(f"Aref_gg: {Aref_gg * 100:.5f}%, Aref_gg_stats_error: {Aref_gg_stats_error * 100:.5f}%")

Aref_gg: 0.0032004607039656796, Aref_gg_stats_error: 0.001343791876758049
Aref_gg: 0.32005%, Aref_gg_stats_error: 0.13438%


In [754]:
Aref_gg_pdg = 0
Acp_etapip_gg_value_3, Acp_etapip_gg_error_3 = correct_Acp_stats_no_Kmix(Araw_gg, Araw_gg_stats_error, Aref_gg, Aref_gg_stats_error, Aref_gg_pdg)

print(f"Acp_etapip_gg_value_3: {Acp_etapip_gg_value_3 * 100:.5f}%, Acp_etapip_gg_error_3: {Acp_etapip_gg_error_3 * 100:.5f}%")

Acp_etapip_gg_value_3: 0.48319%, Acp_etapip_gg_error_3: 0.49637%


In [755]:
x, y, z, x_err, y_err, z_err = Acp_etapip_gg_value_1, Acp_etapip_gg_value_2,  Acp_etapip_gg_value_3,\
                                Acp_etapip_gg_error_1, Acp_etapip_gg_error_2, Acp_etapip_gg_error_3
central_value, error = combine_error_weighted_3(x, y, z, x_err, y_err, z_err)

val 1 = 0.009430448618018894, val 2 = 0.00298948563599738, val 3 = 0.004831931159634928
central value = 0.4704% \pm 0.3623%


In [756]:
original_value = 0.004938869594526052
original_error = 0.003595271697862108

print(f"original_value: {original_value * 100:.5f}% \pm {original_error* 100:.5f}%")


original_value: 0.49389% \pm 0.35953%


In [757]:
(original_value - central_value)

0.00023466806022860633

In [758]:
math.sqrt(abs(error**2 - original_error**2))

0.00044736321035790665

In [759]:
math.sqrt(error**2 - original_error**2)

0.00044736321035790665

In [760]:
(original_value - central_value) / math.sqrt(abs(error**2 - original_error**2))

0.524558244386845

In [761]:
(central_value - original_value) / original_error

-0.06527130073873118

### eta -> pipipi

In [604]:
#bin a, f

In [762]:
#fitv12
Araw_3pi_cms_plus = -0.026349281691045334
Araw_3pi_cms_plus_error = 0.023359224245954303
Araw_3pi_cms_minus = -0.03536209135122792
Araw_3pi_cms_minus_error = 0.01570692416803061

Araw_3pi , Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi: {Araw_3pi}, Araw_3pi_stats_error: {Araw_3pi_stats_error}")
print(f"Araw_3pi: {Araw_3pi * 100:.5f}%, Araw_3pi_stats_error: {Araw_3pi_stats_error * 100:.5f}%")

Araw_3pi: -0.030855686521136627, Araw_3pi_stats_error: 0.01407445224682868
Araw_3pi: -3.08557%, Araw_3pi_stats_error: 1.40745%


In [763]:
#fitv3
Aref_3pi_cms_plus = 0.015942742310851576
Aref_3pi_cms_plus_error =   0.00332833779022883
Aref_3pi_cms_minus = -0.011097528155170178
Aref_3pi_cms_minus_error =  0.002417136926661496
Aref_pipipi, Aref_pipipi_stats_error = combine_x_plus_y_divided_by_2(Aref_3pi_cms_plus,Aref_3pi_cms_minus, Aref_3pi_cms_plus_error,Aref_3pi_cms_minus_error )

print(f"Aref_pipipi: {Aref_pipipi * 100:.5f}%, Aref_pipipi_stats_error: {Aref_pipipi_stats_error * 100:.5f}%")

Aref_pipipi: 0.24226%, Aref_pipipi_stats_error: 0.20567%


In [764]:
Aref_pipipi_pdg = 0
Acp_etapip_pipipi_value_1, Acp_etapip_pipipi_error_1 =correct_Acp_stats_no_Kmix(Araw_3pi, Araw_3pi_stats_error, Aref_pipipi, Aref_pipipi_stats_error, Aref_pipipi_pdg)
print(f"Acp_etapip_pipipi_value_1: {Acp_etapip_pipipi_value_1 * 100:.5f}%, Acp_etapip_pipipi_error_1: {Acp_etapip_pipipi_error_1 * 100:.5f}%")

Acp_etapip_pipipi_value_1: -3.32783%, Acp_etapip_pipipi_error_1: 1.42239%


In [765]:
# bin b, e

In [766]:
#fitv12
Araw_3pi_cms_plus = 0.018310417596028028
Araw_3pi_cms_plus_error = 0.011257977766088875
Araw_3pi_cms_minus = -0.010640451817015673
Araw_3pi_cms_minus_error = 0.009927270894861806

Araw_3pi , Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi: {Araw_3pi}, Araw_3pi_stats_error: {Araw_3pi_stats_error}")
print(f"Araw_3pi: {Araw_3pi * 100:.5f}%, Araw_3pi_stats_error: {Araw_3pi_stats_error * 100:.5f}%")

Araw_3pi: 0.0038349828895061777, Araw_3pi_stats_error: 0.007504877927083854
Araw_3pi: 0.38350%, Araw_3pi_stats_error: 0.75049%


In [767]:
#fitv3
Aref_3pi_cms_plus = 0.008288865417889424
Aref_3pi_cms_plus_error =   0.002099737161326871
Aref_3pi_cms_minus = -0.008329419043432207
Aref_3pi_cms_minus_error =  0.001932319419009154
Aref_pipipi, Aref_pipipi_stats_error = combine_x_plus_y_divided_by_2(Aref_3pi_cms_plus,Aref_3pi_cms_minus, Aref_3pi_cms_plus_error,Aref_3pi_cms_minus_error )

print(f"Aref_pipipi: {Aref_pipipi * 100:.5f}%, Aref_pipipi_stats_error: {Aref_pipipi_stats_error * 100:.5f}%")

Aref_pipipi: -0.00203%, Aref_pipipi_stats_error: 0.14268%


In [768]:
Aref_pipipi_pdg = 0
Acp_etapip_pipipi_value_2, Acp_etapip_pipipi_error_2 =correct_Acp_stats_no_Kmix(Araw_3pi, Araw_3pi_stats_error, Aref_pipipi, Aref_pipipi_stats_error, Aref_pipipi_pdg)
print(f"Acp_etapip_pipipi_value_2: {Acp_etapip_pipipi_value_2 * 100:.5f}%, Acp_etapip_pipipi_error_2: {Acp_etapip_pipipi_error_2 * 100:.5f}%")

Acp_etapip_pipipi_value_2: 0.38553%, Acp_etapip_pipipi_error_2: 0.76393%


In [769]:
# bin c, d

In [770]:
#fitv12
Araw_3pi_cms_plus = 0.004547704899291016
Araw_3pi_cms_plus_error = 0.008475570437081368
Araw_3pi_cms_minus = 0.010569321068468085
Araw_3pi_cms_minus_error = 0.008348121748568953

Araw_3pi , Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi: {Araw_3pi}, Araw_3pi_stats_error: {Araw_3pi_stats_error}")
print(f"Araw_3pi: {Araw_3pi * 100:.5f}%, Araw_3pi_stats_error: {Araw_3pi_stats_error * 100:.5f}%")

Araw_3pi: 0.007558512983879551, Araw_3pi_stats_error: 0.005948244088864746
Araw_3pi: 0.75585%, Araw_3pi_stats_error: 0.59482%


In [771]:
#fitv3
Aref_3pi_cms_plus = 0.007633482723842677
Aref_3pi_cms_plus_error =   0.0017612940547707327
Aref_3pi_cms_minus = -0.00032511575606164467
Aref_3pi_cms_minus_error =  0.0017547755825853024
Aref_pipipi, Aref_pipipi_stats_error = combine_x_plus_y_divided_by_2(Aref_3pi_cms_plus,Aref_3pi_cms_minus, Aref_3pi_cms_plus_error,Aref_3pi_cms_minus_error )

print(f"Aref_pipipi: {Aref_pipipi * 100:.5f}%, Aref_pipipi_stats_error: {Aref_pipipi_stats_error * 100:.5f}%")

Aref_pipipi: 0.36542%, Aref_pipipi_stats_error: 0.12431%


In [772]:
Aref_pipipi_pdg = 0
Acp_etapip_pipipi_value_3, Acp_etapip_pipipi_error_3 =correct_Acp_stats_no_Kmix(Araw_3pi, Araw_3pi_stats_error, Aref_pipipi, Aref_pipipi_stats_error, Aref_pipipi_pdg)
print(f"Acp_etapip_pipipi_value_3: {Acp_etapip_pipipi_value_3 * 100:.5f}%, Acp_etapip_pipipi_error_3: {Acp_etapip_pipipi_error_3 * 100:.5f}%")

Acp_etapip_pipipi_value_3: 0.39043%, Acp_etapip_pipipi_error_3: 0.60768%


In [773]:
x, y, z, x_err, y_err, z_err = Acp_etapip_pipipi_value_1, Acp_etapip_pipipi_value_2, Acp_etapip_pipipi_value_3,\
                                Acp_etapip_pipipi_error_1, Acp_etapip_pipipi_error_2, Acp_etapip_pipipi_error_3
central_value, error = combine_error_weighted_3(x, y, z, x_err, y_err, z_err)

val 1 = -0.033278293598977327, val 2 = 0.003855259702277569, val 3 = 0.0039043294999890343
central value = 0.0149% \pm 0.4510%


In [774]:
original_value = -0.0006098375919060817
original_error = 0.0044942309661693295
print(f"original_value: {original_value * 100:.5f}% \pm {original_error* 100:.5f}%")


original_value: -0.06098% \pm 0.44942%


In [775]:
(original_value - central_value)

-0.0007585233371950142

In [776]:
math.sqrt(abs(error**2 - original_error**2))

0.00037980348998048295

In [777]:
math.sqrt(error**2 - original_error**2)

0.00037980348998048295

In [778]:
(original_value - central_value) / math.sqrt(abs(error**2 - original_error**2))

-1.9971468330477757

In [779]:
(central_value - original_value) / original_error

0.16877711512934185

## Acp(Ds+ -> eta pi+)

### eta -> gg

In [780]:
#bin a, f

In [781]:
#fit_v12
Araw_gg_cms_plus = 0.02054172922314268
Araw_gg_cms_plus_error =  0.010408384273370848
Araw_gg_cms_minus = -0.010052866289589013
Araw_gg_cms_minus_error = 0.01108094106790564
Araw_gg , Araw_gg_stats_error= combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg: {Araw_gg}, Araw_gg_stats_error: {Araw_gg_stats_error}")
print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

Araw_gg: 0.005244431466776833, Araw_gg_stats_error: 0.007601343929407342
Araw_gg: 0.52444%, Araw_gg_stats_error: 0.76013%


In [782]:
#fit_v3
Aref_gg_cms_plus = 0.012767243523722938
Aref_gg_cms_plus_error = 0.0035817861575496578
Aref_gg_cms_minus = -0.009112314113025932
Aref_gg_cms_minus_error = 0.0026362813045371866
Aref_gg, Aref_gg_stats_error = combine_x_plus_y_divided_by_2(Aref_gg_cms_plus,Aref_gg_cms_minus, Aref_gg_cms_plus_error,Aref_gg_cms_minus_error )

print(f"Aref_gg: {Aref_gg}, Aref_gg_stats_error: {Aref_gg_stats_error}")
print(f"Aref_gg: {Aref_gg * 100:.5f}%, Aref_gg_stats_error: {Aref_gg_stats_error * 100:.5f}%")

Aref_gg: 0.0018274647053485027, Aref_gg_stats_error: 0.002223689006755814
Aref_gg: 0.18275%, Aref_gg_stats_error: 0.22237%


In [783]:
Aref_gg_pdg = 0
Acp_etapip_gg_value_1, Acp_etapip_gg_error_1 = correct_Acp_stats_no_Kmix(Araw_gg, Araw_gg_stats_error, Aref_gg, Aref_gg_stats_error, Aref_gg_pdg)

print(f"Acp_etapip_gg_value_1: {Acp_etapip_gg_value_1 * 100:.5f}%, Acp_etapip_gg_error_1: {Acp_etapip_gg_error_1 * 100:.5f}%")

Acp_etapip_gg_value_1: 0.34170%, Acp_etapip_gg_error_1: 0.79199%


In [784]:
# bin b, e

In [785]:
#fit_v12
Araw_gg_cms_plus = 0.015992689534159332
Araw_gg_cms_plus_error =  0.005462259358276075
Araw_gg_cms_minus = -0.008410771408197903
Araw_gg_cms_minus_error = 0.004954885562424106
Araw_gg , Araw_gg_stats_error= combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg: {Araw_gg}, Araw_gg_stats_error: {Araw_gg_stats_error}")
print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

Araw_gg: 0.0037909590629807144, Araw_gg_stats_error: 0.003687382819622659
Araw_gg: 0.37910%, Araw_gg_stats_error: 0.36874%


In [786]:
#fit_v3
Aref_gg_cms_plus = 0.010040423453906655
Aref_gg_cms_plus_error = 0.0022692912068116014
Aref_gg_cms_minus = -0.007031434843011053
Aref_gg_cms_minus_error = 0.002094765417047547
Aref_gg, Aref_gg_stats_error = combine_x_plus_y_divided_by_2(Aref_gg_cms_plus,Aref_gg_cms_minus, Aref_gg_cms_plus_error,Aref_gg_cms_minus_error )

print(f"Aref_gg: {Aref_gg}, Aref_gg_stats_error: {Aref_gg_stats_error}")
print(f"Aref_gg: {Aref_gg * 100:.5f}%, Aref_gg_stats_error: {Aref_gg_stats_error * 100:.5f}%")

Aref_gg: 0.001504494305447801, Aref_gg_stats_error: 0.0015441603490061221
Aref_gg: 0.15045%, Aref_gg_stats_error: 0.15442%


In [787]:
Aref_gg_pdg = 0
Acp_etapip_gg_value_2, Acp_etapip_gg_error_2 = correct_Acp_stats_no_Kmix(Araw_gg, Araw_gg_stats_error, Aref_gg, Aref_gg_stats_error, Aref_gg_pdg)

print(f"Acp_etapip_gg_value_2: {Acp_etapip_gg_value_2 * 100:.5f}%, Acp_etapip_gg_error_2: {Acp_etapip_gg_error_2 * 100:.5f}%")

Acp_etapip_gg_value_2: 0.22865%, Acp_etapip_gg_error_2: 0.39977%


In [788]:
# bin c, d

In [789]:
#fit_v12
Araw_gg_cms_plus = 0.0038186581066048664
Araw_gg_cms_plus_error =  0.004274506688097245
Araw_gg_cms_minus = -0.0006936581187035884
Araw_gg_cms_minus_error = 0.004273349418583739
Araw_gg , Araw_gg_stats_error= combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg: {Araw_gg}, Araw_gg_stats_error: {Araw_gg_stats_error}")
print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

Araw_gg: 0.001562499993950639, Araw_gg_stats_error: 0.0030221235365177437
Araw_gg: 0.15625%, Araw_gg_stats_error: 0.30221%


In [790]:
#fit_v3
Aref_gg_cms_plus = 0.007484012201581125
Aref_gg_cms_plus_error = 0.0019024365813576668
Aref_gg_cms_minus = -0.0010830907936497658
Aref_gg_cms_minus_error = 0.0018983786466550427
Aref_gg, Aref_gg_stats_error = combine_x_plus_y_divided_by_2(Aref_gg_cms_plus,Aref_gg_cms_minus, Aref_gg_cms_plus_error,Aref_gg_cms_minus_error )

print(f"Aref_gg: {Aref_gg}, Aref_gg_stats_error: {Aref_gg_stats_error}")
print(f"Aref_gg: {Aref_gg * 100:.5f}%, Aref_gg_stats_error: {Aref_gg_stats_error * 100:.5f}%")

Aref_gg: 0.0032004607039656796, Aref_gg_stats_error: 0.001343791876758049
Aref_gg: 0.32005%, Aref_gg_stats_error: 0.13438%


In [791]:
Aref_gg_pdg = 0
Acp_etapip_gg_value_3, Acp_etapip_gg_error_3 = correct_Acp_stats_no_Kmix(Araw_gg, Araw_gg_stats_error, Aref_gg, Aref_gg_stats_error, Aref_gg_pdg)

print(f"Acp_etapip_gg_value_3: {Acp_etapip_gg_value_3 * 100:.5f}%, Acp_etapip_gg_error_3: {Acp_etapip_gg_error_3 * 100:.5f}%")

Acp_etapip_gg_value_3: -0.16380%, Acp_etapip_gg_error_3: 0.33074%


In [792]:
x, y, z, x_err, y_err, z_err = Acp_etapip_gg_value_1, Acp_etapip_gg_value_2,  Acp_etapip_gg_value_3,\
                                Acp_etapip_gg_error_1, Acp_etapip_gg_error_2, Acp_etapip_gg_error_3
central_value, error = combine_error_weighted_3(x, y, z, x_err, y_err, z_err)

val 1 = 0.0034169667614283306, val 2 = 0.0022864647575329133, val 3 = -0.0016379607100150406
central value = 0.0281% \pm 0.2426%


In [793]:
original_value = 0.0008267144274838323
original_error = 0.0023568039292432047
print(f"original_value: {original_value * 100:.5f}% \pm {original_error* 100:.5f}%")


original_value: 0.08267% \pm 0.23568%


In [794]:
(original_value - central_value)

0.0005453586372444975

In [795]:
math.sqrt(abs(error**2 - original_error**2))

0.000574616017705946

In [796]:
math.sqrt(error**2 - original_error**2)

0.000574616017705946

In [797]:
(original_value - central_value) / math.sqrt(abs(error**2 - original_error**2))

0.9490835974634793

In [798]:
(central_value - original_value) / original_error

-0.23139754244198757

### eta -> pipipi

In [799]:
#bin a, f

In [800]:
#fitv12
Araw_3pi_cms_plus = -0.0020118595564722064
Araw_3pi_cms_plus_error = 0.014775814345008522
Araw_3pi_cms_minus = -0.03533942595753592
Araw_3pi_cms_minus_error = 0.01570630756421268

Araw_3pi , Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi: {Araw_3pi}, Araw_3pi_stats_error: {Araw_3pi_stats_error}")
print(f"Araw_3pi: {Araw_3pi * 100:.5f}%, Araw_3pi_stats_error: {Araw_3pi_stats_error * 100:.5f}%")

Araw_3pi: -0.018675642757004063, Araw_3pi_stats_error: 0.010782077569510945
Araw_3pi: -1.86756%, Araw_3pi_stats_error: 1.07821%


In [801]:
#fitv3
Aref_3pi_cms_plus = 0.015942742310851576
Aref_3pi_cms_plus_error =   0.00332833779022883
Aref_3pi_cms_minus = -0.011097528155170178
Aref_3pi_cms_minus_error =  0.002417136926661496
Aref_pipipi, Aref_pipipi_stats_error = combine_x_plus_y_divided_by_2(Aref_3pi_cms_plus,Aref_3pi_cms_minus, Aref_3pi_cms_plus_error,Aref_3pi_cms_minus_error )

print(f"Aref_pipipi: {Aref_pipipi * 100:.5f}%, Aref_pipipi_stats_error: {Aref_pipipi_stats_error * 100:.5f}%")

Aref_pipipi: 0.24226%, Aref_pipipi_stats_error: 0.20567%


In [802]:
Aref_pipipi_pdg = 0
Acp_etapip_pipipi_value_1, Acp_etapip_pipipi_error_1 =correct_Acp_stats_no_Kmix(Araw_3pi, Araw_3pi_stats_error, Aref_pipipi, Aref_pipipi_stats_error, Aref_pipipi_pdg)
print(f"Acp_etapip_pipipi_value_1: {Acp_etapip_pipipi_value_1 * 100:.5f}%, Acp_etapip_pipipi_error_1: {Acp_etapip_pipipi_error_1 * 100:.5f}%")

Acp_etapip_pipipi_value_1: -2.10982%, Acp_etapip_pipipi_error_1: 1.09765%


In [803]:
# bin b, e

In [804]:
#fitv12
Araw_3pi_cms_plus = 0.025503290180761473
Araw_3pi_cms_plus_error = 0.0074159382959272145
Araw_3pi_cms_minus = -0.004980864839452148
Araw_3pi_cms_minus_error = 0.006615879311427364

Araw_3pi , Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi: {Araw_3pi}, Araw_3pi_stats_error: {Araw_3pi_stats_error}")
print(f"Araw_3pi: {Araw_3pi * 100:.5f}%, Araw_3pi_stats_error: {Araw_3pi_stats_error * 100:.5f}%")

Araw_3pi: 0.010261212670654662, Araw_3pi_stats_error: 0.004969054232758293
Araw_3pi: 1.02612%, Araw_3pi_stats_error: 0.49691%


In [805]:
#fitv3
Aref_3pi_cms_plus = 0.008288865417889424
Aref_3pi_cms_plus_error =   0.002099737161326871
Aref_3pi_cms_minus = -0.008329419043432207
Aref_3pi_cms_minus_error =  0.001932319419009154 
Aref_pipipi, Aref_pipipi_stats_error = combine_x_plus_y_divided_by_2(Aref_3pi_cms_plus,Aref_3pi_cms_minus, Aref_3pi_cms_plus_error,Aref_3pi_cms_minus_error )

print(f"Aref_pipipi: {Aref_pipipi * 100:.5f}%, Aref_pipipi_stats_error: {Aref_pipipi_stats_error * 100:.5f}%")

Aref_pipipi: -0.00203%, Aref_pipipi_stats_error: 0.14268%


In [806]:
Aref_pipipi_pdg = 0
Acp_etapip_pipipi_value_2, Acp_etapip_pipipi_error_2 =correct_Acp_stats_no_Kmix(Araw_3pi, Araw_3pi_stats_error, Aref_pipipi, Aref_pipipi_stats_error, Aref_pipipi_pdg)
print(f"Acp_etapip_pipipi_value_2: {Acp_etapip_pipipi_value_2 * 100:.5f}%, Acp_etapip_pipipi_error_2: {Acp_etapip_pipipi_error_2 * 100:.5f}%")

Acp_etapip_pipipi_value_2: 1.02815%, Acp_etapip_pipipi_error_2: 0.51698%


In [807]:
# bin c, d

In [808]:
#fitv12
Araw_3pi_cms_plus = -0.005295325231744363
Araw_3pi_cms_plus_error = 0.005618524064807555
Araw_3pi_cms_minus = 0.0032871804901470902
Araw_3pi_cms_minus_error = 0.00561428192409719

Araw_3pi , Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi: {Araw_3pi}, Araw_3pi_stats_error: {Araw_3pi_stats_error}")
print(f"Araw_3pi: {Araw_3pi * 100:.5f}%, Araw_3pi_stats_error: {Araw_3pi_stats_error * 100:.5f}%")

Araw_3pi: -0.0010040723707986365, Araw_3pi_stats_error: 0.003971396926462591
Araw_3pi: -0.10041%, Araw_3pi_stats_error: 0.39714%


In [809]:
#fitv3
Aref_3pi_cms_plus = 0.007633482723842677
Aref_3pi_cms_plus_error =   0.0017612940547707327
Aref_3pi_cms_minus = -0.00032511575606164467
Aref_3pi_cms_minus_error =  0.0017547755825853024
Aref_pipipi, Aref_pipipi_stats_error = combine_x_plus_y_divided_by_2(Aref_3pi_cms_plus,Aref_3pi_cms_minus, Aref_3pi_cms_plus_error,Aref_3pi_cms_minus_error )

print(f"Aref_pipipi: {Aref_pipipi * 100:.5f}%, Aref_pipipi_stats_error: {Aref_pipipi_stats_error * 100:.5f}%")

Aref_pipipi: 0.36542%, Aref_pipipi_stats_error: 0.12431%


In [810]:
Aref_pipipi_pdg = 0
Acp_etapip_pipipi_value_3, Acp_etapip_pipipi_error_3 =correct_Acp_stats_no_Kmix(Araw_3pi, Araw_3pi_stats_error, Aref_pipipi, Aref_pipipi_stats_error, Aref_pipipi_pdg)
print(f"Acp_etapip_pipipi_value_3: {Acp_etapip_pipipi_value_3 * 100:.5f}%, Acp_etapip_pipipi_error_3: {Acp_etapip_pipipi_error_3 * 100:.5f}%")

Acp_etapip_pipipi_value_3: -0.46583%, Acp_etapip_pipipi_error_3: 0.41614%


In [811]:
x, y, z, x_err, y_err, z_err = Acp_etapip_pipipi_value_1, Acp_etapip_pipipi_value_2, Acp_etapip_pipipi_value_3,\
                                Acp_etapip_pipipi_error_1, Acp_etapip_pipipi_error_2, Acp_etapip_pipipi_error_3
central_value, error = combine_error_weighted_3(x, y, z, x_err, y_err, z_err)

val 1 = -0.021098249834844762, val 2 = 0.010281489483426054, val 3 = -0.004658255854689153
central value = -0.0574% \pm 0.3109%


In [812]:
original_value = -0.0005604291899353742
original_error = 0.00302622050106648
print(f"original_value: {original_value * 100:.5f}% \pm {original_error* 100:.5f}%")


original_value: -0.05604% \pm 0.30262%


In [813]:
(original_value - central_value)

1.3938367103502776e-05

In [814]:
math.sqrt(abs(error**2 - original_error**2))

0.00071239545301059

In [815]:
math.sqrt(error**2 - original_error**2)

0.00071239545301059

In [816]:
(original_value - central_value) / math.sqrt(abs(error**2 - original_error**2))

0.01956549139189351

In [817]:
(central_value - original_value) / original_error

-0.004605866326855796

## Acp(D+ -> eta K+)

### eta -> gg

In [818]:
#bin a, f

In [819]:
#fit_v12
Araw_gg_cms_plus = 0.16756031741801514
Araw_gg_cms_plus_error = 0.2864336869193095
Araw_gg_cms_minus = 0.24040281977084033
Araw_gg_cms_minus_error = 0.18144112899285988
Araw_gg , Araw_gg_stats_error= combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg: {Araw_gg}, Araw_gg_stats_error: {Araw_gg_stats_error}")
print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

Araw_gg: 0.20398156859442773, Araw_gg_stats_error: 0.16953254871291873
Araw_gg: 20.39816%, Araw_gg_stats_error: 16.95325%


In [820]:
#fit_v3
Aref_gg_cms_plus = 0.017252864854357952
Aref_gg_cms_plus_error = 0.007918940763228284
Aref_gg_cms_minus = 0.016508283372701937
Aref_gg_cms_minus_error = 0.005484386206955928
Aref_gg, Aref_gg_stats_error = combine_x_plus_y_divided_by_2(Aref_gg_cms_plus,Aref_gg_cms_minus, Aref_gg_cms_plus_error,Aref_gg_cms_minus_error )

print(f"Aref_gg: {Aref_gg}, Aref_gg_stats_error: {Aref_gg_stats_error}")
print(f"Aref_gg: {Aref_gg * 100:.5f}%, Aref_gg_stats_error: {Aref_gg_stats_error * 100:.5f}%")

Aref_gg: 0.016880574113529945, Aref_gg_stats_error: 0.004816329382386731
Aref_gg: 1.68806%, Aref_gg_stats_error: 0.48163%


In [821]:
Aref_gg_pdg = 0
Acp_etapip_gg_value_1, Acp_etapip_gg_error_1 = correct_Acp_stats_no_Kmix(Araw_gg, Araw_gg_stats_error, Aref_gg, Aref_gg_stats_error, Aref_gg_pdg)

print(f"Acp_etapip_gg_value_1: {Acp_etapip_gg_value_1 * 100:.5f}%, Acp_etapip_gg_error_1: {Acp_etapip_gg_error_1 * 100:.5f}%")

Acp_etapip_gg_value_1: 18.71010%, Acp_etapip_gg_error_1: 16.96009%


In [822]:
# bin b, e

In [823]:
#fit_v12
Araw_gg_cms_plus =  0.261650258048318
Araw_gg_cms_plus_error = 0.11465407567241663
Araw_gg_cms_minus = 0.0491789531678557
Araw_gg_cms_minus_error = 0.10411678013072416
Araw_gg , Araw_gg_stats_error= combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg: {Araw_gg}, Araw_gg_stats_error: {Araw_gg_stats_error}")
print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

Araw_gg: 0.15541460560808684, Araw_gg_stats_error: 0.07743684680610032
Araw_gg: 15.54146%, Araw_gg_stats_error: 7.74368%


In [824]:
#fit_v3
Aref_gg_cms_plus = 0.0220502302141643
Aref_gg_cms_plus_error = 0.004398569789016606
Aref_gg_cms_minus = -0.014824985123424117
Aref_gg_cms_minus_error = 0.00433340895975646
Aref_gg, Aref_gg_stats_error = combine_x_plus_y_divided_by_2(Aref_gg_cms_plus,Aref_gg_cms_minus, Aref_gg_cms_plus_error,Aref_gg_cms_minus_error )

print(f"Aref_gg: {Aref_gg}, Aref_gg_stats_error: {Aref_gg_stats_error}")
print(f"Aref_gg: {Aref_gg * 100:.5f}%, Aref_gg_stats_error: {Aref_gg_stats_error * 100:.5f}%")

Aref_gg: 0.003612622545370092, Aref_gg_stats_error: 0.003087306649870853
Aref_gg: 0.36126%, Aref_gg_stats_error: 0.30873%


In [825]:
Aref_gg_pdg = 0
Acp_etapip_gg_value_2, Acp_etapip_gg_error_2 = correct_Acp_stats_no_Kmix(Araw_gg, Araw_gg_stats_error, Aref_gg, Aref_gg_stats_error, Aref_gg_pdg)

print(f"Acp_etapip_gg_value_2: {Acp_etapip_gg_value_2 * 100:.5f}%, Acp_etapip_gg_error_2: {Acp_etapip_gg_error_2 * 100:.5f}%")

Acp_etapip_gg_value_2: 15.18020%, Acp_etapip_gg_error_2: 7.74984%


In [826]:
# bin c, d

In [827]:
#fit_v12
Araw_gg_cms_plus = -0.004122162206123048
Araw_gg_cms_plus_error = 0.07643209429512954
Araw_gg_cms_minus = 0.10990470561184629
Araw_gg_cms_minus_error = 0.09372667565462571
Araw_gg , Araw_gg_stats_error= combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg: {Araw_gg}, Araw_gg_stats_error: {Araw_gg_stats_error}")
print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

Araw_gg: 0.05289127170286162, Araw_gg_stats_error: 0.060470147113280165
Araw_gg: 5.28913%, Araw_gg_stats_error: 6.04701%


In [828]:
#fit_v3
Aref_gg_cms_plus = 0.012187909643263106
Aref_gg_cms_plus_error = 0.0037722126382702466
Aref_gg_cms_minus = 0.0018652971803998497
Aref_gg_cms_minus_error = 0.003992315064765103
Aref_gg, Aref_gg_stats_error = combine_x_plus_y_divided_by_2(Aref_gg_cms_plus,Aref_gg_cms_minus, Aref_gg_cms_plus_error,Aref_gg_cms_minus_error )

print(f"Aref_gg: {Aref_gg}, Aref_gg_stats_error: {Aref_gg_stats_error}")
print(f"Aref_gg: {Aref_gg * 100:.5f}%, Aref_gg_stats_error: {Aref_gg_stats_error * 100:.5f}%")

Aref_gg: 0.007026603411831478, Aref_gg_stats_error: 0.0027462778339361516
Aref_gg: 0.70266%, Aref_gg_stats_error: 0.27463%


In [829]:
Aref_gg_pdg = 0
Acp_etapip_gg_value_3, Acp_etapip_gg_error_3 = correct_Acp_stats_no_Kmix(Araw_gg, Araw_gg_stats_error, Aref_gg, Aref_gg_stats_error, Aref_gg_pdg)

print(f"Acp_etapip_gg_value_3: {Acp_etapip_gg_value_3 * 100:.5f}%, Acp_etapip_gg_error_3: {Acp_etapip_gg_error_3 * 100:.5f}%")

Acp_etapip_gg_value_3: 4.58647%, Acp_etapip_gg_error_3: 6.05325%


In [830]:
x, y, z, x_err, y_err, z_err = Acp_etapip_gg_value_1, Acp_etapip_gg_value_2,  Acp_etapip_gg_value_3,\
                                Acp_etapip_gg_error_1, Acp_etapip_gg_error_2, Acp_etapip_gg_error_3
central_value, error = combine_error_weighted_3(x, y, z, x_err, y_err, z_err)

val 1 = 0.1871009944808978, val 2 = 0.15180198306271675, val 3 = 0.04586466829103014
central value = 9.3418% \pm 4.5923%


In [831]:
original_value = 0.09423048411790164
original_error = 0.045257554265262276
print(f"original_value: {original_value * 100:.5f}% \pm {original_error* 100:.5f}%")


original_value: 9.42305% \pm 4.52576%


In [832]:
(original_value - central_value)

0.0008125740585086944

In [833]:
math.sqrt(abs(error**2 - original_error**2))

0.007788966260399373

In [834]:
math.sqrt(error**2 - original_error**2)

0.007788966260399373

In [835]:
(original_value - central_value) / math.sqrt(abs(error**2 - original_error**2))

0.10432373582614932

In [836]:
(central_value - original_value) / original_error

-0.01795444035146174

### eta -> pipipi

In [837]:
#bin a, f

In [838]:
#fitv12
Araw_3pi_cms_plus = -0.2535369314552338
Araw_3pi_cms_plus_error = 0.22130685122206725
Araw_3pi_cms_minus = -0.40881744883739646
Araw_3pi_cms_minus_error = 0.2329800115892308

Araw_3pi , Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi: {Araw_3pi}, Araw_3pi_stats_error: {Araw_3pi_stats_error}")
print(f"Araw_3pi: {Araw_3pi * 100:.5f}%, Araw_3pi_stats_error: {Araw_3pi_stats_error * 100:.5f}%")

Araw_3pi: -0.33117719014631514, Araw_3pi_stats_error: 0.16066767580781793
Araw_3pi: -33.11772%, Araw_3pi_stats_error: 16.06677%


In [839]:
#fitv3
# Aref_3pi_cms_plus = 0.023243885002202536
# Aref_3pi_cms_plus_error = 0.006711540436810791
Aref_3pi_cms_plus = 0.023217944925122636
Aref_3pi_cms_plus_error = 0.0067097809366402665
Aref_3pi_cms_minus = 0.015218142947458935
Aref_3pi_cms_minus_error = 0.004537685540033986
Aref_pipipi, Aref_pipipi_stats_error = combine_x_plus_y_divided_by_2(Aref_3pi_cms_plus,Aref_3pi_cms_minus, Aref_3pi_cms_plus_error,Aref_3pi_cms_minus_error )

print(f"Aref_pipipi: {Aref_pipipi * 100:.5f}%, Aref_pipipi_stats_error: {Aref_pipipi_stats_error * 100:.5f}%")

Aref_pipipi: 1.92180%, Aref_pipipi_stats_error: 0.40501%


In [840]:
Aref_pipipi_pdg = 0
Acp_etapip_pipipi_value_1, Acp_etapip_pipipi_error_1 =correct_Acp_stats_no_Kmix(Araw_3pi, Araw_3pi_stats_error, Aref_pipipi, Aref_pipipi_stats_error, Aref_pipipi_pdg)
print(f"Acp_etapip_pipipi_value_1: {Acp_etapip_pipipi_value_1 * 100:.5f}%, Acp_etapip_pipipi_error_1: {Acp_etapip_pipipi_error_1 * 100:.5f}%")

Acp_etapip_pipipi_value_1: -35.03952%, Acp_etapip_pipipi_error_1: 16.07187%


In [841]:
# bin b, e

In [842]:
#fitv12
Araw_3pi_cms_plus = 0.11156614048734892
Araw_3pi_cms_plus_error = 0.10666691158049672
Araw_3pi_cms_minus = 0.08985996357087433
Araw_3pi_cms_minus_error = 0.10807293659176577

Araw_3pi , Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi: {Araw_3pi}, Araw_3pi_stats_error: {Araw_3pi_stats_error}")
print(f"Araw_3pi: {Araw_3pi * 100:.5f}%, Araw_3pi_stats_error: {Araw_3pi_stats_error * 100:.5f}%")

Araw_3pi: 0.10071305202911163, Araw_3pi_stats_error: 0.07592362881489749
Araw_3pi: 10.07131%, Araw_3pi_stats_error: 7.59236%


In [843]:
#fitv3
Aref_3pi_cms_plus = 0.02125550433722756
Aref_3pi_cms_plus_error = 0.003710636783933779
Aref_3pi_cms_minus = -0.010137816529805388
Aref_3pi_cms_minus_error = 0.0036231797252456147
Aref_pipipi, Aref_pipipi_stats_error = combine_x_plus_y_divided_by_2(Aref_3pi_cms_plus,Aref_3pi_cms_minus, Aref_3pi_cms_plus_error,Aref_3pi_cms_minus_error )

print(f"Aref_pipipi: {Aref_pipipi * 100:.5f}%, Aref_pipipi_stats_error: {Aref_pipipi_stats_error * 100:.5f}%")

Aref_pipipi: 0.55588%, Aref_pipipi_stats_error: 0.25931%


In [844]:
Aref_pipipi_pdg = 0
Acp_etapip_pipipi_value_2, Acp_etapip_pipipi_error_2 =correct_Acp_stats_no_Kmix(Araw_3pi, Araw_3pi_stats_error, Aref_pipipi, Aref_pipipi_stats_error, Aref_pipipi_pdg)
print(f"Acp_etapip_pipipi_value_2: {Acp_etapip_pipipi_value_2 * 100:.5f}%, Acp_etapip_pipipi_error_2: {Acp_etapip_pipipi_error_2 * 100:.5f}%")

Acp_etapip_pipipi_value_2: 9.51542%, Acp_etapip_pipipi_error_2: 7.59679%


In [845]:
# bin c, d

In [846]:
#fitv12
Araw_3pi_cms_plus = 0.0748696651939289
Araw_3pi_cms_plus_error = 0.08291304365895245
Araw_3pi_cms_minus = -0.006401176656638974
Araw_3pi_cms_minus_error = 0.08078364591151699


Araw_3pi , Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi: {Araw_3pi}, Araw_3pi_stats_error: {Araw_3pi_stats_error}")
print(f"Araw_3pi: {Araw_3pi * 100:.5f}%, Araw_3pi_stats_error: {Araw_3pi_stats_error * 100:.5f}%")

Araw_3pi: 0.034234244268644964, Araw_3pi_stats_error: 0.057880416065256284
Araw_3pi: 3.42342%, Araw_3pi_stats_error: 5.78804%


In [847]:
#fitv3
Aref_3pi_cms_plus = 0.0132798278277928
Aref_3pi_cms_plus_error = 0.003172959523596482
Aref_3pi_cms_minus = 0.0006751803797835354
Aref_3pi_cms_minus_error = 0.003352792139958763
Aref_pipipi, Aref_pipipi_stats_error = combine_x_plus_y_divided_by_2(Aref_3pi_cms_plus,Aref_3pi_cms_minus, Aref_3pi_cms_plus_error,Aref_3pi_cms_minus_error )

print(f"Aref_pipipi: {Aref_pipipi * 100:.5f}%, Aref_pipipi_stats_error: {Aref_pipipi_stats_error * 100:.5f}%")

Aref_pipipi: 0.69775%, Aref_pipipi_stats_error: 0.23081%


In [848]:
Aref_pipipi_pdg = 0
Acp_etapip_pipipi_value_3, Acp_etapip_pipipi_error_3 =correct_Acp_stats_no_Kmix(Araw_3pi, Araw_3pi_stats_error, Aref_pipipi, Aref_pipipi_stats_error, Aref_pipipi_pdg)
print(f"Acp_etapip_pipipi_value_3: {Acp_etapip_pipipi_value_3 * 100:.5f}%, Acp_etapip_pipipi_error_3: {Acp_etapip_pipipi_error_3 * 100:.5f}%")

Acp_etapip_pipipi_value_3: 2.72567%, Acp_etapip_pipipi_error_3: 5.79264%


In [849]:
x, y, z, x_err, y_err, z_err = Acp_etapip_pipipi_value_1, Acp_etapip_pipipi_value_2, Acp_etapip_pipipi_value_3,\
                                Acp_etapip_pipipi_error_1, Acp_etapip_pipipi_error_2, Acp_etapip_pipipi_error_3
central_value, error = combine_error_weighted_3(x, y, z, x_err, y_err, z_err)

val 1 = -0.3503952340826059, val 2 = 0.09515420812540054, val 3 = 0.027256740164856796
central value = 2.1658% \pm 4.4280%


In [850]:
original_value = 0.0031169401297088672
original_error = 0.045016438283892596
print(f"original_value: {original_value * 100:.5f}% \pm {original_error* 100:.5f}%")


original_value: 0.31169% \pm 4.50164%


In [851]:
(original_value - central_value)

-0.018541202739589036

In [852]:
math.sqrt(abs(error**2 - original_error**2))

0.00810795554606171

In [853]:
math.sqrt(error**2 - original_error**2)

ValueError: math domain error

In [854]:
(original_value - central_value) / math.sqrt(abs(error**2 - original_error**2))

-2.2867913661163426

In [855]:
(central_value - original_value) / original_error

0.4118762711225711

## Acp(Ds+ -> eta K+)

### eta -> gg

In [856]:
#bin a, f

In [857]:
#fit_v12
Araw_gg_cms_plus = 0.07708835828379579
Araw_gg_cms_plus_error =  0.05804554330420574
Araw_gg_cms_minus = 0.06364143061147298
Araw_gg_cms_minus_error = 0.03759720005500946
Araw_gg , Araw_gg_stats_error= combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg: {Araw_gg}, Araw_gg_stats_error: {Araw_gg_stats_error}")
print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

Araw_gg: 0.07036489444763439, Araw_gg_stats_error: 0.034579020190922224
Araw_gg: 7.03649%, Araw_gg_stats_error: 3.45790%


In [858]:
#fit_v3
Aref_gg_cms_plus = 0.017252864854357952
Aref_gg_cms_plus_error = 0.007918940763228284
Aref_gg_cms_minus = 0.016508283372701937
Aref_gg_cms_minus_error = 0.005484386206955928
Aref_gg, Aref_gg_stats_error = combine_x_plus_y_divided_by_2(Aref_gg_cms_plus,Aref_gg_cms_minus, Aref_gg_cms_plus_error,Aref_gg_cms_minus_error )

print(f"Aref_gg: {Aref_gg}, Aref_gg_stats_error: {Aref_gg_stats_error}")
print(f"Aref_gg: {Aref_gg * 100:.5f}%, Aref_gg_stats_error: {Aref_gg_stats_error * 100:.5f}%")

Aref_gg: 0.016880574113529945, Aref_gg_stats_error: 0.004816329382386731
Aref_gg: 1.68806%, Aref_gg_stats_error: 0.48163%


In [859]:
Aref_gg_pdg = 0
Acp_etapip_gg_value_1, Acp_etapip_gg_error_1 = correct_Acp_stats_no_Kmix(Araw_gg, Araw_gg_stats_error, Aref_gg, Aref_gg_stats_error, Aref_gg_pdg)

print(f"Acp_etapip_gg_value_1: {Acp_etapip_gg_value_1 * 100:.5f}%, Acp_etapip_gg_error_1: {Acp_etapip_gg_error_1 * 100:.5f}%")

Acp_etapip_gg_value_1: 5.34843%, Acp_etapip_gg_error_1: 3.49128%


In [860]:
# bin b, e

In [861]:
#fit_v12
Araw_gg_cms_plus =  0.03610012071435431
Araw_gg_cms_plus_error = 0.026345808372576458
Araw_gg_cms_minus = 0.04037604703681286
Araw_gg_cms_minus_error = 0.0247619866318869
Araw_gg , Araw_gg_stats_error= combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg: {Araw_gg}, Araw_gg_stats_error: {Araw_gg_stats_error}")
print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

Araw_gg: 0.038238083875583584, Araw_gg_stats_error: 0.018078008745173412
Araw_gg: 3.82381%, Araw_gg_stats_error: 1.80780%


In [862]:
#fit_v3
Aref_gg_cms_plus = 0.0220502302141643
Aref_gg_cms_plus_error = 0.004398569789016606
Aref_gg_cms_minus = -0.014824985123424117
Aref_gg_cms_minus_error = 0.00433340895975646
Aref_gg, Aref_gg_stats_error = combine_x_plus_y_divided_by_2(Aref_gg_cms_plus,Aref_gg_cms_minus, Aref_gg_cms_plus_error,Aref_gg_cms_minus_error )

print(f"Aref_gg: {Aref_gg}, Aref_gg_stats_error: {Aref_gg_stats_error}")
print(f"Aref_gg: {Aref_gg * 100:.5f}%, Aref_gg_stats_error: {Aref_gg_stats_error * 100:.5f}%")

Aref_gg: 0.003612622545370092, Aref_gg_stats_error: 0.003087306649870853
Aref_gg: 0.36126%, Aref_gg_stats_error: 0.30873%


In [863]:
Aref_gg_pdg = 0
Acp_etapip_gg_value_2, Acp_etapip_gg_error_2 = correct_Acp_stats_no_Kmix(Araw_gg, Araw_gg_stats_error, Aref_gg, Aref_gg_stats_error, Aref_gg_pdg)

print(f"Acp_etapip_gg_value_2: {Acp_etapip_gg_value_2 * 100:.5f}%, Acp_etapip_gg_error_2: {Acp_etapip_gg_error_2 * 100:.5f}%")

Acp_etapip_gg_value_2: 3.46255%, Acp_etapip_gg_error_2: 1.83397%


In [864]:
# bin c, d

In [865]:
#fit_v12
Araw_gg_cms_plus = 0.04413298877670302
Araw_gg_cms_plus_error =  0.020310443962151137
Araw_gg_cms_minus = 0.010653423278667029
Araw_gg_cms_minus_error = 0.022040722939479424
Araw_gg , Araw_gg_stats_error= combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg: {Araw_gg}, Araw_gg_stats_error: {Araw_gg_stats_error}")
print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

Araw_gg: 0.027393206027685024, Araw_gg_stats_error: 0.01498589004392612
Araw_gg: 2.73932%, Araw_gg_stats_error: 1.49859%


In [866]:
#fit_v3
Aref_gg_cms_plus = 0.012187909643263106
Aref_gg_cms_plus_error = 0.0037722126382702466
Aref_gg_cms_minus = 0.0018652971803998497
Aref_gg_cms_minus_error = 0.003992315064765103
Aref_gg, Aref_gg_stats_error = combine_x_plus_y_divided_by_2(Aref_gg_cms_plus,Aref_gg_cms_minus, Aref_gg_cms_plus_error,Aref_gg_cms_minus_error )

print(f"Aref_gg: {Aref_gg}, Aref_gg_stats_error: {Aref_gg_stats_error}")
print(f"Aref_gg: {Aref_gg * 100:.5f}%, Aref_gg_stats_error: {Aref_gg_stats_error * 100:.5f}%")

Aref_gg: 0.007026603411831478, Aref_gg_stats_error: 0.0027462778339361516
Aref_gg: 0.70266%, Aref_gg_stats_error: 0.27463%


In [867]:
Aref_gg_pdg = 0
Acp_etapip_gg_value_3, Acp_etapip_gg_error_3 = correct_Acp_stats_no_Kmix(Araw_gg, Araw_gg_stats_error, Aref_gg, Aref_gg_stats_error, Aref_gg_pdg)

print(f"Acp_etapip_gg_value_3: {Acp_etapip_gg_value_3 * 100:.5f}%, Acp_etapip_gg_error_3: {Acp_etapip_gg_error_3 * 100:.5f}%")

Acp_etapip_gg_value_3: 2.03666%, Acp_etapip_gg_error_3: 1.52355%


In [868]:
x, y, z, x_err, y_err, z_err = Acp_etapip_gg_value_1, Acp_etapip_gg_value_2,  Acp_etapip_gg_value_3,\
                                Acp_etapip_gg_error_1, Acp_etapip_gg_error_2, Acp_etapip_gg_error_3
central_value, error = combine_error_weighted_3(x, y, z, x_err, y_err, z_err)

val 1 = 0.05348432033410444, val 2 = 0.03462546133021349, val 3 = 0.020366602615853546
central value = 2.8953% \pm 1.1110%


In [869]:
original_value = 0.029623108305721846
original_error = 0.01099655679281541

In [870]:
(original_value - central_value)

0.0006701960741839312

In [871]:
math.sqrt(abs(error**2 - original_error**2))

0.0015833559854844838

In [872]:
math.sqrt(error**2 - original_error**2)

0.0015833559854844838

In [873]:
(original_value - central_value) / math.sqrt(abs(error**2 - original_error**2))

0.42327567541853894

In [874]:
(central_value - original_value) / original_error

-0.060945993078651964

### eta -> pipipi

In [875]:
#bin a, f

In [876]:
#fitv12
Araw_3pi_cms_plus = 0.1370534319567649
Araw_3pi_cms_plus_error = 0.07730209429788548
Araw_3pi_cms_minus = 0.022989424431700822
Araw_3pi_cms_minus_error = 0.048716947074588124

Araw_3pi , Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi: {Araw_3pi}, Araw_3pi_stats_error: {Araw_3pi_stats_error}")
print(f"Araw_3pi: {Araw_3pi * 100:.5f}%, Araw_3pi_stats_error: {Araw_3pi_stats_error * 100:.5f}%")

Araw_3pi: 0.08002142819423286, Araw_3pi_stats_error: 0.04568630734450804
Araw_3pi: 8.00214%, Araw_3pi_stats_error: 4.56863%


In [877]:
#fitv3
# Aref_3pi_cms_plus = 0.023243885002202536
# Aref_3pi_cms_plus_error = 0.006711540436810791
Aref_3pi_cms_plus = 0.023217944925122636
Aref_3pi_cms_plus_error = 0.0067097809366402665
Aref_3pi_cms_minus = 0.015218142947458935
Aref_3pi_cms_minus_error = 0.004537685540033986
Aref_pipipi, Aref_pipipi_stats_error = combine_x_plus_y_divided_by_2(Aref_3pi_cms_plus,Aref_3pi_cms_minus, Aref_3pi_cms_plus_error,Aref_3pi_cms_minus_error )

print(f"Aref_pipipi: {Aref_pipipi * 100:.5f}%, Aref_pipipi_stats_error: {Aref_pipipi_stats_error * 100:.5f}%")

Aref_pipipi: 1.92180%, Aref_pipipi_stats_error: 0.40501%


In [878]:
Aref_pipipi_pdg = 0
Acp_etapip_pipipi_value_1, Acp_etapip_pipipi_error_1 =correct_Acp_stats_no_Kmix(Araw_3pi, Araw_3pi_stats_error, Aref_pipipi, Aref_pipipi_stats_error, Aref_pipipi_pdg)
print(f"Acp_etapip_pipipi_value_1: {Acp_etapip_pipipi_value_1 * 100:.5f}%, Acp_etapip_pipipi_error_1: {Acp_etapip_pipipi_error_1 * 100:.5f}%")

Acp_etapip_pipipi_value_1: 6.08034%, Acp_etapip_pipipi_error_1: 4.58655%


In [879]:
# bin b, e

In [880]:
#fitv12
Araw_3pi_cms_plus = 0.023253317328598255
Araw_3pi_cms_plus_error = 0.029860433239910375
Araw_3pi_cms_minus = 0.004681952208063667
Araw_3pi_cms_minus_error = 0.030058448436172377

Araw_3pi , Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi: {Araw_3pi}, Araw_3pi_stats_error: {Araw_3pi_stats_error}")
print(f"Araw_3pi: {Araw_3pi * 100:.5f}%, Araw_3pi_stats_error: {Araw_3pi_stats_error * 100:.5f}%")

Araw_3pi: 0.013967634768330961, Araw_3pi_stats_error: 0.021184639456839818
Araw_3pi: 1.39676%, Araw_3pi_stats_error: 2.11846%


In [881]:
#fitv3
Aref_3pi_cms_plus = 0.02125550433722756
Aref_3pi_cms_plus_error = 0.003710636783933779
Aref_3pi_cms_minus = -0.010137816529805388
Aref_3pi_cms_minus_error = 0.0036231797252456147
Aref_pipipi, Aref_pipipi_stats_error = combine_x_plus_y_divided_by_2(Aref_3pi_cms_plus,Aref_3pi_cms_minus, Aref_3pi_cms_plus_error,Aref_3pi_cms_minus_error )

print(f"Aref_pipipi: {Aref_pipipi * 100:.5f}%, Aref_pipipi_stats_error: {Aref_pipipi_stats_error * 100:.5f}%")

Aref_pipipi: 0.55588%, Aref_pipipi_stats_error: 0.25931%


In [882]:
Aref_pipipi_pdg = 0
Acp_etapip_pipipi_value_2, Acp_etapip_pipipi_error_2 =correct_Acp_stats_no_Kmix(Araw_3pi, Araw_3pi_stats_error, Aref_pipipi, Aref_pipipi_stats_error, Aref_pipipi_pdg)
print(f"Acp_etapip_pipipi_value_2: {Acp_etapip_pipipi_value_2 * 100:.5f}%, Acp_etapip_pipipi_error_2: {Acp_etapip_pipipi_error_2 * 100:.5f}%")

Acp_etapip_pipipi_value_2: 0.84088%, Acp_etapip_pipipi_error_2: 2.13428%


In [883]:
# bin c, d

In [884]:
#fitv12
Araw_3pi_cms_plus = 0.033827364566805684
Araw_3pi_cms_plus_error = 0.024925581664785024
Araw_3pi_cms_minus = -0.006401176656638974
Araw_3pi_cms_minus_error = 0.08078364591151699

Araw_3pi , Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi: {Araw_3pi}, Araw_3pi_stats_error: {Araw_3pi_stats_error}")
print(f"Araw_3pi: {Araw_3pi * 100:.5f}%, Araw_3pi_stats_error: {Araw_3pi_stats_error * 100:.5f}%")

Araw_3pi: 0.013713093955083355, Araw_3pi_stats_error: 0.042270799815254334
Araw_3pi: 1.37131%, Araw_3pi_stats_error: 4.22708%


In [885]:
#fitv3
Aref_3pi_cms_plus = 0.0132798278277928
Aref_3pi_cms_plus_error = 0.003172959523596482
Aref_3pi_cms_minus = 0.0006751803797835354
Aref_3pi_cms_minus_error = 0.003352792139958763
Aref_pipipi, Aref_pipipi_stats_error = combine_x_plus_y_divided_by_2(Aref_3pi_cms_plus,Aref_3pi_cms_minus, Aref_3pi_cms_plus_error,Aref_3pi_cms_minus_error )

print(f"Aref_pipipi: {Aref_pipipi * 100:.5f}%, Aref_pipipi_stats_error: {Aref_pipipi_stats_error * 100:.5f}%")

Aref_pipipi: 0.69775%, Aref_pipipi_stats_error: 0.23081%


In [886]:
Aref_pipipi_pdg = 0
Acp_etapip_pipipi_value_3, Acp_etapip_pipipi_error_3 =correct_Acp_stats_no_Kmix(Araw_3pi, Araw_3pi_stats_error, Aref_pipipi, Aref_pipipi_stats_error, Aref_pipipi_pdg)
print(f"Acp_etapip_pipipi_value_3: {Acp_etapip_pipipi_value_3 * 100:.5f}%, Acp_etapip_pipipi_error_3: {Acp_etapip_pipipi_error_3 * 100:.5f}%")

Acp_etapip_pipipi_value_3: 0.67356%, Acp_etapip_pipipi_error_3: 4.23338%


In [887]:
x, y, z, x_err, y_err, z_err = Acp_etapip_pipipi_value_1, Acp_etapip_pipipi_value_2, Acp_etapip_pipipi_value_3,\
                                Acp_etapip_pipipi_error_1, Acp_etapip_pipipi_error_2, Acp_etapip_pipipi_error_3
central_value, error = combine_error_weighted_3(x, y, z, x_err, y_err, z_err)

val 1 = 0.06080338425794207, val 2 = 0.008408790864619875, val 3 = 0.006735589851295187
central value = 1.5834% \pm 1.7599%


In [888]:
original_value = 0.008337253666614142
original_error = 0.013088513903116376

In [889]:
(original_value - central_value)

-0.007496542422130416

In [890]:
math.sqrt(abs(error**2 - original_error**2))

0.011764974767203716

In [891]:
math.sqrt(error**2 - original_error**2)

0.011764974767203716

In [892]:
(original_value - central_value) / math.sqrt(abs(error**2 - original_error**2))

-0.6371915427330903

In [893]:
(original_value - central_value) / original_error

-0.5727573411023759

In [894]:
(central_value - original_value) / original_error

0.5727573411023759